In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import xarray as xr
from glob import glob
import os
from netCDF4 import Dataset
import pandas as pd
from datetime import datetime, date, timedelta
from pathlib import Path
import scipy
import scipy.ndimage
from mpl_toolkits.axes_grid1 import ImageGrid
import math

#variables
selected_time = 335000
negative_w_threshold = -0.25

In [ ]:
#Open datasets
ds_ql = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/ql.nc", decode_times=False).isel(time=slice(1,None))
ds_w = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/w.nc", decode_times=False).isel(time=slice(1,None)).rename({'zh':'z'}).interp(z=ds_ql.z)

#cleaning data
ds_ql = ds_ql.fillna(0)
ds_w = ds_w.fillna(0)

#ds_w.w.sel(time=selected_time, method="nearest").plot.surface(x="x",y="y")


In [ ]:
#creating binary mask for ql
ds_ql_binary = ds_ql.where(ds_ql.ql == 0, 1)

ds_ql_binary_renamed = ds_ql_binary.rename_vars(ql="w") #renaming ql so it can be used

#plot
#ds_ql_binary.ql.sel(time=selected_time, method="nearest").plot.surface(x="x",y="y")

In [ ]:
#creating binary mask for w
ds_w_binary = ds_w.where(ds_w.w < negative_w_threshold, 0).where(ds_w.w >= negative_w_threshold, 1)


plt.show()

In [ ]:
ql_mask = ds_ql_binary_renamed.w.values.astype(bool)
w_mask = ds_w_binary.w.values.astype(bool)

outline_mask = np.zeros_like(ql_mask)

#creating array for expansion
expansion = np.zeros((3,3,3), dtype=bool)
expansion[1, 1, :] = True  # X axis
expansion[1, :, 1] = True  # Y axis
expansion[:, 1, 1] = True  # Z axis

for t in range(ql_mask.shape[0]):
    current = ql_mask[t, :, :, :]
    w_slice = w_mask[t, :, :, :]

    slices = scipy.ndimage.find_objects(w_slice)
    if not slices:
        continue

    #obtain a boundary box and apply it to ql and w
    bbox = slices[0]
    sub_current = current[bbox].copy()
    sub_w = w_slice[bbox].copy()

    front = sub_current.copy()

    while np.any(front):
        dilated_front = scipy.ndimage.binary_dilation(front,structure=expansion, mask=sub_w)
        next_front = dilated_front & ~sub_current

        sub_current |= next_front
        front = next_front

    outline_mask[t][bbox] = sub_current & ~ql_mask[t][bbox]

ds_ql_shift_whole = ds_ql_binary_renamed.copy()
ds_ql_shift_whole["w"] = (ds_ql_binary_renamed["w"].dims, outline_mask.astype(float))

In [ ]:
#combining the two masks
ds_shell_mask = ds_ql_shift_whole * ds_w_binary

In [ ]:
#shell w can now be obtained by multiplying the graph and the mask
ds_shell_mask = ds_shell_mask.w.where(ds_shell_mask.w != 0,np.nan)
ds_w_shell = ds_w * ds_shell_mask

In [ ]:
#adding contours of ql
#ds_w_shell_contour = xr.merge([ds_w_shell,ds_ql])


#plotting
df = ds_w_shell.sel(time=selected_time, method="nearest").to_dataframe().reset_index()
df_active = df.dropna(subset=['w'])

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(df_active['x'],df_active['y'],df_active['z'],c=df_active['w'],cmap='viridis',s=10,alpha=0.6)

ax.set_xlabel('X')
ax.set_ylabel('Y')
fig.colorbar(scatter,ax=ax,label='W Value',shrink=0.6)
